# 02 Limpieza de los datos

**Proyecto:** Análisis de estrategias de inversión en el S&P 500

En el notebook anterior (01_exploracion) revisé los datos y vi que están bastante limpios: los tipos de datos son correctos y no hay nulos ni filas duplicadas. Aun así, encontré algunas cosas que tengo que corregir antes de empezar el análisis.

Decidí usar solo las empresas con datos desde el primer día (4 de enero de 2016), para que todas tengan los mismos años de datos y se puedan comparar de forma justa. Así pierdo 39 empresas de 503.

También vi que había días con volumen 0, es decir, días en los que no se compró ni vendió ninguna acción. Casi todos eran de SW, FERG y AMCR, que tenían precios de relleno de antes de empezar a cotizar en EE. UU., así que también las quito. Con esto me quedan 461 empresas.

### Qué voy a hacer

- Me quedo solo con las 461 empresas.

- Borro las filas sueltas con volumen 0.

- Quito el último día si lo descargué con la bolsa abierta.

- Paso los sectores al español.

- Dejo la fecha sin la hora.

### Datos

Parto de los archivos de Data/Raw (empresas.csv, precios.csv e indice.csv) y guardaré los datos limpios en Data/Processed para usarlos en los siguientes notebooks.

## 1. Cargar los datos

Primero conecto Google Drive, porque ahí tengo guardados los CSV generados en el notebook anterior.

In [2]:
# Importo pandas para trabajar con tablas
import pandas as pd
# Importo os para poder crear carpetas si no existen
import os

# Compruebo dónde se está ejecutando el notebook
try:
    # Si estoy en Colab, conecto mi Drive y uso mi carpeta
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_PROYECTO = "/content/drive/MyDrive/PORTFOLIO MARCOS GUERRERO SANCHEZ/analisis-estrategias-inversion/"
except ImportError:
    # Si no estoy en Colab, uso la carpeta donde esté el proyecto descargado
    # El ../ sube un nivel, porque el notebook está dentro de Notebooks/
    RUTA_PROYECTO = "../"

# Carpeta con los datos originales (los que descargué en el notebook 00)
RUTA = RUTA_PROYECTO + "Data/Raw/"
# Carpeta donde voy a guardar los datos ya limpios
RUTA_PROCESSED = RUTA_PROYECTO + "Data/Processed/"

# Si la carpeta de datos limpios no existe, la creo
os.makedirs(RUTA_PROCESSED, exist_ok=True)

# Cargo los tres archivos originales
empresas = pd.read_csv(RUTA + "empresas.csv")
precios = pd.read_csv(RUTA + "precios.csv", parse_dates=["Date"])
indice = pd.read_csv(RUTA + "indice.csv", parse_dates=["Date"])

print("Empresas:", empresas.shape)
print("Precios:", precios.shape)
print("Índice:", indice.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Empresas: (503, 4)
Precios: (1315666, 7)
Índice: (2708, 6)


## 2. Empresas válidas

En la exploración decidí quedarme solo con las empresas que tienen datos desde el primer día y que no tienen muchos días con volumen 0. Aquí calculo cuáles son.

En vez de escribir la fecha y los nombres de las empresas a mano, lo hago con reglas, para que si algún día actualizo los datos siga funcionando igual.

In [3]:
# Primera fecha de cada empresa
# Separo las filas por empresa (Ticker) y coge la fecha más pequeña de cada una
primera_fecha = precios.groupby("ticker")["Date"].min()

# Busco la fecha más antigua de toda la tabla
fecha_inicio_datos = precios["Date"].min()

print("Primera fecha de los datos:",fecha_inicio_datos.date())

Primera fecha de los datos: 2016-01-04


In [4]:
# Si una empresa tiene más días con volumen 0 que este número
# (Decisión própia ya que las empresas con problemas tenían cientos de días y las demás solo uno o dos)
MAX_DIAS_VOLUMEN_CERO = 20

# Me quedo solo con las filas en las que el volumen es 0
volumen_cero = precios[precios["Volume"] == 0]

# Cuento cuántos días con volumen 0 tiene cada empresa
dias_volumen_cero = volumen_cero["ticker"].value_counts()

# Me quedo con las empresas que superan el número que he puesto arriba
sospechosas = dias_volumen_cero[dias_volumen_cero > MAX_DIAS_VOLUMEN_CERO].index

print("Empresas sospechosas:", list(sospechosas))

Empresas sospechosas: ['SW', 'FERG', 'AMCR', 'HWM']


In [5]:
# Me quedo con las empresas que empiezan el primer día de los datos
empresas_desde_2016 = primera_fecha[primera_fecha == fecha_inicio_datos].index

# Lista vacía donde guardaré las empresas que sí voy a usar
empresas_finales = []

# Recorro las empresas que empiezan el primer día
for ticker in empresas_desde_2016:
    # Si no está entre las sospechosas, la añado a la lista
    if ticker not in sospechosas:
        empresas_finales.append(ticker)

print("Empresas que usaré en el análisis:", len(empresas_finales))

Empresas que usaré en el análisis: 461


Con estas dos reglas me quedan 462 empresas, las mismas que había contado a mano en la exploración.

## 3. Filtro la tabla de precios

Ya tengo la lista de las 462 empresas. Ahora me quedo solo con sus filas en la tabla de precios y quito las demás.

In [6]:
#Al quitar esas 41 empresas he eliminado también todas sus filas de precios: de 1.309.545 filas paso a 1.243.241, es decir, unas 66.000 filas menos.
# Me quedo con 462 empresas, todas con precios desde enero de 2016 hasta hoy, así que ya puedo compararlas entre ellas.
# isin() mira en cada fila si el ticker está dentro de mi lista
# .copy() hace una copia independiente, porque luego voy a seguir modificando esta tabla
precios_limpios = precios[precios["ticker"].isin(empresas_finales)].copy()

# Compruebo cuántas filas tenía antes y cuántas me quedan
print("Filas antes:", len(precios))
print("Filas después:", len(precios_limpios))

# Compruebo que quedan 462 empresas distintas
print("Empresas distintas:", precios_limpios["ticker"].nunique())

Filas antes: 1315666
Filas después: 1248387
Empresas distintas: 461


## 4. Borro las filas con volumen 0

Después de quitar las empresas sospechosas, todavía quedan algunos días sueltos con volumen 0 en otras empresas. Son muy pocos y parecen fallos puntuales, así que borro esas filas.

In [7]:
# Guardo cuántas filas tengo antes de borrar, para poder comparar
filas_antes = len(precios_limpios)

# Me quedo con las filas en las que el volumen no es 0
precios_limpios = precios_limpios[precios_limpios["Volume"] != 0]

print("Filas borradas:", filas_antes - len(precios_limpios))
print("Filas que quedan:", len(precios_limpios))

Filas borradas: 15
Filas que quedan: 1248372


Solo he borrado 15 filas, porque la mayoría de los días con volumen 0 se fueron al quitar las empresas sospechosas. Aun así dejo este paso en el código, para que si actualizo los datos y aparecen días nuevos con volumen 0, se borren solos.

## 5. Traduzco los sectores al español

Los sectores vienen en inglés desde Wikipedia. Los paso al español para que el dashboard de Power BI se entienda mejor.

In [8]:
# Diccionario con la traducción de cada sector
# A la izquierda el nombre en inglés (Clave) y a la derecha (Valor) cómo quiero que aparezca
traduccion_sectores = {
    "Information Technology": "Tecnología",
    "Financials": "Financiero",
    "Health Care": "Salud",
    "Consumer Discretionary": "Consumo discrecional",
    "Industrials": "Industrial",
    "Consumer Staples": "Consumo básico",
    "Energy": "Energía",
    "Utilities": "Servicios públicos",
    "Real Estate": "Inmobiliario",
    "Materials": "Materiales",
    "Communication Services": "Comunicaciones",
}

# Hago una copia de la tabla de empresas para no modificar la original
# copy() crea una tabla nueva e independiente: si cambio la copia, la original se queda igual
empresas_limpias = empresas.copy()

# Cambio cada sector en inglés por su traducción
# replace() busca el valor de cada fila entre las claves del diccionario y lo sustituye
# por su valor. Si un sector no está en el diccionario, lo deja como estaba
empresas_limpias["sector"] = empresas_limpias["sector"].replace(traduccion_sectores)

# Me quedo solo con las 462 empresas que voy a usar, igual que hice con los precios
empresas_limpias = empresas_limpias[empresas_limpias["ticker"].isin(empresas_finales)]

# Compruebo cuántas empresas quedan y cuántas hay de cada sector
# value_counts() cuenta cuántas empresas hay de cada sector, así veo si alguno sigue en inglés
print("Empresas:", len(empresas_limpias))
print(empresas_limpias["sector"].value_counts())

Empresas: 461
sector
Financiero              74
Industrial              70
Tecnología              68
Salud                   57
Consumo discrecional    44
Consumo básico          32
Servicios públicos      29
Inmobiliario            28
Materiales              21
Energía                 20
Comunicaciones          18
Name: count, dtype: int64


## 6. Dejo la fecha sin la hora

Las fechas llevan la hora puesta a 00:00:00, que no sirve de nada porque los datos son diarios. La quito para que las fechas queden más limpias en SQL y en Power BI.

In [9]:
# Me aseguro de que la columna es de tipo fecha
# pd.to_datetime() convierte los valores en fechas. Si ya lo eran, no cambia nada
# Así la celda funciona aunque la ejecute más de una vez
precios_limpios["Date"] = pd.to_datetime(precios_limpios["Date"])

# Quito la hora de la columna de fechas
# .dt sirve para trabajar con columnas de fecha, y .date deja solo el día
precios_limpios["Date"] = precios_limpios["Date"].dt.date

# Compruebo cómo han quedado las fechas
print(precios_limpios["Date"].head())

0    2016-01-04
1    2016-01-05
2    2016-01-06
3    2016-01-07
4    2016-01-08
Name: Date, dtype: object


## 7. Así quedan las tablas

Antes de guardar, reviso cómo han quedado las tablas después de la limpieza.

In [10]:
# Compruebo el tamaño de la tabla de empresas
# shape devuelve (filas, columnas). No lleva paréntesis porque es una característica de la tabla
print("Tamaño de empresas:", empresas_limpias.shape)

# Muestro las primeras filas para ver cómo ha quedado
# head() enseña las 5 primeras filas de la tabla
empresas_limpias.head()

Tamaño de empresas: (461, 4)


,ticker,nombre,sector,industria
0,MMM,3M,Industrial,Industrial Conglomerates
1,AOS,A. O. Smith,Industrial,Building Products
2,ABT,Abbott Laboratories,Salud,Health Care Equipment
3,ABBV,AbbVie,Salud,Biotechnology
4,ACN,Accenture,Tecnología,IT Consulting & Other Services


In [11]:
# Compruebo el tamaño de la tabla de precios
print("Tamaño de precios:", precios_limpios.shape)

# Compruebo que no hay valores vacíos en ninguna columna
# isnull() marca con True las celdas vacías y sum() las cuenta, porque True vale 1
print("\nValores nulos por columna:")
print(precios_limpios.isnull().sum())

# Muestro las primeras filas para ver cómo ha quedado
precios_limpios.head()

Tamaño de precios: (1248372, 7)

Valores nulos por columna:
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
ticker    0
dtype: int64


,Date,Close,High,Low,Open,Volume,ticker
0,2016-01-04,87.046112,87.935427,86.204231,87.775355,3919531,MMM
1,2016-01-05,87.425552,87.449269,86.328732,87.046111,3214968,MMM
2,2016-01-06,85.664711,86.417665,85.030335,86.316874,3584532,MMM
3,2016-01-07,83.577797,84.858412,83.376220,84.496758,4249986,MMM
4,2016-01-08,83.293213,84.484895,83.133141,83.809015,3186144,MMM


In [12]:
# Busco el último día que tengo en los datos
# max() devuelve la fecha más reciente de la columna
ultimo_dia = precios_limpios["Date"].max()

# Me quedo solo con los precios de ese último día (una fila por empresa)
ultimos_precios = precios_limpios[precios_limpios["Date"] == ultimo_dia]

# Junto los precios con la información de las empresas
# merge() une dos tablas usando una columna que tienen en común, en este caso el ticker
resumen = ultimos_precios.merge(empresas_limpias, on="ticker")

# Muestro las columnas que me interesan de las 10 primeras empresas
# Con doble corchete elijo varias columnas a la vez
print("Precios del día:", ultimo_dia)
resumen[["ticker", "nombre", "sector", "Close", "Volume"]].head(10)



Precios del día: 2026-10-09


,ticker,nombre,sector,Close,Volume
0,MMM,3M,Industrial,159.960007,2372200
1,AOS,A. O. Smith,Industrial,56.380001,2264700
2,ABT,Abbott Laboratories,Salud,99.599998,6961900
3,ABBV,AbbVie,Salud,276.470001,4297400
4,ACN,Accenture,Tecnología,209.059998,4876700
5,ADBE,Adobe Inc.,Tecnología,242.270004,3889700
6,AMD,Advanced Micro Devices,Tecnología,608.099976,13694400
7,AES,AES Corporation,Servicios públicos,14.940000,5586100
8,AFL,Aflac,Financiero,114.540001,1477100
9,A,Agilent Technologies,Salud,170.539993,1537300


## 8. Compruebo el último día

Al revisar los datos vi que el último día tenía mucho menos volumen que los anteriores. Esto es debido a que en el momento de descargar los datos, no se habia cerrado la bolsa todavía, aún le faltaban varias horas y por tanto los datos no estaban completos. Lo compruebo sumando el volumen de todas las empresas por día.

In [13]:
volumen_por_dia = precios_limpios.groupby("Date")["Volume"].sum()
volumen_por_dia.tail()

,Volume
Date,
2026-10-05,2640385362
2026-10-06,2501263401
2026-10-07,2441633969
2026-10-08,3247408417
2026-10-09,2401672597


La primera vez que descargué los datos, el último día tenía menos de la mitad de volumen que los anteriores, porque lo descargué con la bolsa todavía abierta. Para solucionarlo cambié el notebook de descarga para que solo baje días ya cerrados, y volví a ejecutar los notebooks.

Ahora el último día tiene un volumen normal. Dejo esta comprobación para revisarla cada vez que actualice los datos: si el último día sale con muy poco volumen, algo ha fallado en la descarga.

El 18 de septiembre tiene casi el doble de volumen que un día normal, pero no es un error: es el tercer viernes de septiembre, un día en que vencen muchos contratos financieros y el índice ajusta sus empresas, así que hay mucha más actividad.

## 9. Limpio la tabla del índice

La tabla del índice (SPY) tiene una sola serie de precios, así que no hay empresas que quitar. Solo compruebo que no tiene valores vacíos y le quito la hora a la fecha, igual que a la tabla de precios.

In [14]:
# Hago una copia de la tabla del índice para no modificar la original
# copy() crea una tabla nueva e independiente: si cambio la copia, la original se queda igual
indice_limpio = indice.copy()

# Me aseguro de que la columna es de tipo fecha
# pd.to_datetime() convierte los valores en fechas. Si ya lo eran, no cambia nada
indice_limpio["Date"] = pd.to_datetime(indice_limpio["Date"])

# Quito la hora de la columna de fechas
# .dt sirve para trabajar con columnas de fecha, y .date deja solo el día
indice_limpio["Date"] = indice_limpio["Date"].dt.date

# Compruebo que no hay valores vacíos en ninguna columna
# isnull() marca con True las celdas vacías y sum() las cuenta, porque True vale 1
print("Valores nulos por columna:")
print(indice_limpio.isnull().sum())

# Compruebo el tamaño y cómo ha quedado la tabla
# shape devuelve (filas, columnas) y head() muestra las 5 primeras filas
print("\nTamaño del índice:", indice_limpio.shape)
indice_limpio.head()

Valores nulos por columna:
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

Tamaño del índice: (2708, 6)


,Date,Close,High,Low,Open,Volume
0,2016-01-04,169.051758,169.060163,167.008194,168.606045,222353500
1,2016-01-05,169.337692,169.791810,168.236024,169.371325,110845800
2,2016-01-06,167.201660,168.244455,166.175676,166.797985,152112600
3,2016-01-07,163.190247,166.041135,162.803395,164.266687,213436100
4,2016-01-08,161.398895,164.703913,161.112969,164.148870,209817200


La tabla del índice no tiene valores vacíos y ya tiene la fecha sin la hora. Tiene el mismo número de días que las empresas, así que puedo compararlas día a día.

## 10. Guardo los datos limpios

Guardo las tres tablas limpias en la carpeta Data/Processed. A partir de ahora, los notebooks de análisis leerán estos archivos en lugar de los originales.

In [15]:
# Guardo las tres tablas como archivos CSV
# to_csv() convierte la tabla en un archivo CSV
# index=False evita que se añada una columna extra con la numeración de las filas
empresas_limpias.to_csv(RUTA_PROCESSED + "empresas_limpias.csv", index=False)
precios_limpios.to_csv(RUTA_PROCESSED + "precios_limpios.csv", index=False)
indice_limpio.to_csv(RUTA_PROCESSED + "indice_limpio.csv", index=False)

# Confirmo el guardado mostrando el tamaño de cada tabla
print("Empresas:", empresas_limpias.shape)
print("Precios:", precios_limpios.shape)
print("Índice:", indice_limpio.shape)

Empresas: (461, 4)
Precios: (1248372, 7)
Índice: (2708, 6)


In [16]:
# Compruebo cuántas empresas quedan después de cada filtro, para entender el número final
print("Empresas en la lista de Wikipedia:", empresas["ticker"].nunique())
print("Empresas con precios descargados:", precios["ticker"].nunique())
print("Empresas con datos desde 2016:", len(empresas_desde_2016))
print("Empresas descartadas por precios de relleno:", len(sospechosas))
print("Empresas finales:", len(empresas_finales))

# Veo qué empresas se han descartado y cuántos días con volumen cero tenían
print("\nDetalle de las descartadas:")
print(dias_volumen_cero[dias_volumen_cero > MAX_DIAS_VOLUMEN_CERO])

Empresas en la lista de Wikipedia: 503
Empresas con precios descargados: 503
Empresas con datos desde 2016: 464
Empresas descartadas por precios de relleno: 4
Empresas finales: 461

Detalle de las descartadas:
ticker
SW      1172
FERG     838
AMCR     669
HWM       35
Name: count, dtype: int64


## 11. Resumen de la limpieza

He dejado los datos listos para el análisis:

Me quedo con las empresas que tienen datos desde el primer día y sin días de relleno. Lo calculo con reglas y no a mano, para que funcione igual si actualizo los datos.

He borrado las filas sueltas con volumen 0, he pasado los sectores al español y he quitado la hora de las fechas.

He comprobado que el último día descargado está completo.

Las tres tablas limpias están guardadas en Data/Processed: empresas_limpias.csv, precios_limpios.csv e indice_limpio.csv.